# 🥉 Notebook 01 — Bronze Ingestion

**Project:** Energy Fraud & Default Risk Detection  
**Layer:** Bronze (Raw → Delta)  
**Author:** Zara Louise  
**Stack:** PySpark + Delta Lake + Unity Catalog

---

## 🎯 Goal

Read raw CSV files from ANEEL (SAMP + Inadimplência datasets) from the volume `energy_project.raw.aneel_files` and persist them as **Delta tables** in the `energy_project.bronze` schema, with no business transformations — only:

- Reading with explicit schema enforcement
- Adding ingestion metadata (timestamp, source file)
- Idempotent persistence in Delta format

## 📊 Tables ingested

| Delta Table | Source | Approx. Size |
|---|---|---|
| `energy_project.bronze.samp` | `samp/samp-*.csv` (2020-2026) | ~1.5 GB |
| `energy_project.bronze.inadimplencia` | `inadimplencia/inadimplencia.csv` | ~58 MB |
| `energy_project.bronze.dominio_indicadores` | `inadimplencia/dominio-indicadores.csv` | ~40 KB |

## 🏛️ Medallion Architecture Context

This notebook implements the **Bronze layer** of the medallion architecture:

- **Raw** (Volume) → CSV files as-is from ANEEL portal
- **Bronze** (this layer) → Delta tables, minimal transformation, schema enforced
- **Silver** → Cleansed, deduplicated, joined data
- **Gold** → Business aggregations, ML features, BI-ready

In [0]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
# In Databricks, the SparkSession (`spark`) is already instantiated natively.
# No need to create one manually as we would in local PySpark.

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType

print("✅ Libraries imported — ready for Bronze ingestion!")

✅ Libraries imported — ready for Bronze ingestion!


In [0]:
# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================
# Centralizing paths and table names in variables makes maintenance easier.
# Best practice: if something changes, update only here — no need to hunt 
# through the entire codebase.

# Catalog and schemas
CATALOG = "energy_project"
SCHEMA_BRONZE = "bronze"

# Source paths (raw files in the Volume)
VOLUME_BASE = "/Volumes/energy_project/raw/aneel_files"
PATH_SAMP = f"{VOLUME_BASE}/samp"
PATH_INADIMPLENCIA = f"{VOLUME_BASE}/inadimplencia"

# Destination table names (Bronze layer)
TABLE_SAMP = f"{CATALOG}.{SCHEMA_BRONZE}.samp"
TABLE_INADIMPLENCIA = f"{CATALOG}.{SCHEMA_BRONZE}.inadimplencia"
TABLE_DOMAIN_INDICATORS = f"{CATALOG}.{SCHEMA_BRONZE}.dominio_indicadores"

print("📂 Source paths:")
print(f"   SAMP:            {PATH_SAMP}")
print(f"   Inadimplência:   {PATH_INADIMPLENCIA}")
print()
print("🎯 Destination tables:")
print(f"   {TABLE_SAMP}")
print(f"   {TABLE_INADIMPLENCIA}")
print(f"   {TABLE_DOMAIN_INDICATORS}")

📂 Source paths:
   SAMP:            /Volumes/energy_project/raw/aneel_files/samp
   Inadimplência:   /Volumes/energy_project/raw/aneel_files/inadimplencia

🎯 Destination tables:
   energy_project.bronze.samp
   energy_project.bronze.inadimplencia
   energy_project.bronze.dominio_indicadores


In [0]:
# =============================================================================
# SAMP — READ ALL CSV FILES WITH EXPLICIT STRING SCHEMA
# =============================================================================
# Bronze layer philosophy: preserve 100% of source data as raw strings.
# Type casting is deferred to the Silver layer with proper error handling.
#
# Why explicit string schema instead of inferSchema?
# 1. Deterministic — same result every time, regardless of cluster state
# 2. Lossless — no rows dropped due to type inference failures
# 3. Source-faithful — CSVs ARE strings; we preserve that nature
# 4. Best practice — Databricks-recommended pattern for Bronze layer

# Define the schema explicitly — all 18 columns as StringType
# Column names match ANEEL's data dictionary (preserved for traceability)
SAMP_SCHEMA = StructType([
    StructField("DatGeracaoConjuntoDados",    StringType(), True),
    StructField("NumCNPJAgenteDistribuidora", StringType(), True),
    StructField("SigAgenteDistribuidora",     StringType(), True),
    StructField("NomAgenteDistribuidora",     StringType(), True),
    StructField("NomTipoMercado",             StringType(), True),
    StructField("DscModalidadeTarifaria",     StringType(), True),
    StructField("DscSubGrupoTarifario",       StringType(), True),
    StructField("DscClasseConsumoMercado",    StringType(), True),
    StructField("DscSubClasseConsumidor",     StringType(), True),
    StructField("DscDetalheConsumidor",       StringType(), True),
    StructField("IdeAgenteAcessante",         StringType(), True),
    StructField("NumCNPJAgenteAcessante",     StringType(), True),
    StructField("NomAgenteAcessante",         StringType(), True),
    StructField("DscPostoTarifario",          StringType(), True),
    StructField("DscOpcaoEnergia",            StringType(), True),
    StructField("DscDetalheMercado",          StringType(), True),
    StructField("DatCompetencia",             StringType(), True),
    StructField("VlrMercado",                 StringType(), True),
])

# Read all SAMP CSV files with the explicit schema
df_samp_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .schema(SAMP_SCHEMA)                  # 🔑 Explicit schema, NOT inferSchema
    .csv(f"{PATH_SAMP}/*.csv")
)

print(f"✅ SAMP raw data loaded with explicit string schema")
print(f"   📊 Columns: {len(df_samp_raw.columns)}")
print(f"   🔢 Total rows: {df_samp_raw.count():,}")
print(f"\n📋 Schema (all StringType — type casting deferred to Silver):")
df_samp_raw.printSchema()

✅ SAMP raw data loaded with explicit string schema
   📊 Columns: 18
   🔢 Total rows: 6,808,591

📋 Schema (all StringType — type casting deferred to Silver):
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- NumCNPJAgenteDistribuidora: string (nullable = true)
 |-- SigAgenteDistribuidora: string (nullable = true)
 |-- NomAgenteDistribuidora: string (nullable = true)
 |-- NomTipoMercado: string (nullable = true)
 |-- DscModalidadeTarifaria: string (nullable = true)
 |-- DscSubGrupoTarifario: string (nullable = true)
 |-- DscClasseConsumoMercado: string (nullable = true)
 |-- DscSubClasseConsumidor: string (nullable = true)
 |-- DscDetalheConsumidor: string (nullable = true)
 |-- IdeAgenteAcessante: string (nullable = true)
 |-- NumCNPJAgenteAcessante: string (nullable = true)
 |-- NomAgenteAcessante: string (nullable = true)
 |-- DscPostoTarifario: string (nullable = true)
 |-- DscOpcaoEnergia: string (nullable = true)
 |-- DscDetalheMercado: string (nullable = true)
 |-- 

In [0]:
# =============================================================================
# VISUAL CHECK — Confirm Portuguese accents render correctly
# =============================================================================
# Expected: 'Compensação', 'Não se aplica' (proper accents)
# If broken: 'Compensa��o' → wrong encoding, try Windows-1252

print("🔍 Sample row (vertical view):")
df_samp_raw.show(1, vertical=True, truncate=False)

🔍 Sample row (vertical view):
-RECORD 0------------------------------------------------------------------------------
 DatGeracaoConjuntoDados    | 2025-10-20                                               
 NumCNPJAgenteDistribuidora | 61695227000193                                           
 SigAgenteDistribuidora     | ELETROPAULO                                              
 NomAgenteDistribuidora     | ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO PAULO S.A. 
 NomTipoMercado             | Regular                                                  
 DscModalidadeTarifaria     | Branca                                                   
 DscSubGrupoTarifario       | B3                                                       
 DscClasseConsumoMercado    | Industrial                                               
 DscSubClasseConsumidor     | Não se aplica                                            
 DscDetalheConsumidor       | Não se aplica                                            
 I

In [0]:
# =============================================================================
# COLUMN DICTIONARY (PT-BR → EN reference)
# =============================================================================
# Original Portuguese names preserved in Bronze for full traceability 
# to the ANEEL data dictionary. Translation applied in the Silver layer.

SAMP_COLUMN_DICTIONARY = {
    "DatGeracaoConjuntoDados":    "dataset_generation_date",
    "NumCNPJAgenteDistribuidora": "distributor_cnpj",
    "SigAgenteDistribuidora":     "distributor_code",
    "NomAgenteDistribuidora":     "distributor_name",
    "NomTipoMercado":             "market_type",
    "DscModalidadeTarifaria":     "tariff_modality",
    "DscSubGrupoTarifario":       "tariff_subgroup",
    "DscClasseConsumoMercado":    "consumption_class",
    "DscSubClasseConsumidor":     "consumer_subclass",
    "DscDetalheConsumidor":       "consumer_detail",
    "IdeAgenteAcessante":         "accessing_agent_id",
    "NumCNPJAgenteAcessante":     "accessing_agent_cnpj",
    "NomAgenteAcessante":         "accessing_agent_name",
    "DscPostoTarifario":          "tariff_period",
    "DscOpcaoEnergia":            "energy_option",
    "DscDetalheMercado":          "market_detail",
    "DatCompetencia":             "reference_date",
    "VlrMercado":                 "market_value",
}

print(f"📖 SAMP column dictionary: {len(SAMP_COLUMN_DICTIONARY)} columns mapped\n")
for pt, en in SAMP_COLUMN_DICTIONARY.items():
    print(f"   {pt:32} → {en}")

📖 SAMP column dictionary: 18 columns mapped

   DatGeracaoConjuntoDados          → dataset_generation_date
   NumCNPJAgenteDistribuidora       → distributor_cnpj
   SigAgenteDistribuidora           → distributor_code
   NomAgenteDistribuidora           → distributor_name
   NomTipoMercado                   → market_type
   DscModalidadeTarifaria           → tariff_modality
   DscSubGrupoTarifario             → tariff_subgroup
   DscClasseConsumoMercado          → consumption_class
   DscSubClasseConsumidor           → consumer_subclass
   DscDetalheConsumidor             → consumer_detail
   IdeAgenteAcessante               → accessing_agent_id
   NumCNPJAgenteAcessante           → accessing_agent_cnpj
   NomAgenteAcessante               → accessing_agent_name
   DscPostoTarifario                → tariff_period
   DscOpcaoEnergia                  → energy_option
   DscDetalheMercado                → market_detail
   DatCompetencia                   → reference_date
   VlrMercado       

In [0]:
# =============================================================================
# ADD INGESTION METADATA
# =============================================================================
# Audit columns to track WHEN and FROM WHERE each row was loaded.
# Critical for debugging, lineage, and incremental processing.
#
# Note: Unity Catalog deprecates input_file_name(), recommending the modern
# _metadata.file_path syntax instead.

df_samp_bronze = (
    df_samp_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingestion_date", F.current_date())
)

print("✅ Metadata columns added:")
print("   • _ingestion_timestamp → load time")
print("   • _source_file → original CSV path (via _metadata)")
print("   • _ingestion_date → date for partitioning\n")

df_samp_bronze.select(
    "DatCompetencia", "SigAgenteDistribuidora", "VlrMercado",
    "_ingestion_timestamp", "_source_file"
).show(3, truncate=False)

✅ Metadata columns added:
   • _ingestion_timestamp → load time
   • _source_file → original CSV path (via _metadata)
   • _ingestion_date → date for partitioning

+--------------+----------------------+-------------+--------------------------+---------------------------------------------------------------+
|DatCompetencia|SigAgenteDistribuidora|VlrMercado   |_ingestion_timestamp      |_source_file                                                   |
+--------------+----------------------+-------------+--------------------------+---------------------------------------------------------------+
|2020-10-01    |ELETROPAULO           |76,000000    |2026-05-17 22:55:46.701749|dbfs:/Volumes/energy_project/raw/aneel_files/samp/samp-2020.csv|
|2020-10-01    |ELETROPAULO           |266754,610000|2026-05-17 22:55:46.701749|dbfs:/Volumes/energy_project/raw/aneel_files/samp/samp-2020.csv|
|2020-10-01    |ELETROPAULO           |386352,000000|2026-05-17 22:55:46.701749|dbfs:/Volumes/energy_project/ra

In [0]:
# =============================================================================
# WRITE TO BRONZE LAYER AS DELTA TABLE (with partitioning)
# =============================================================================
# Following best practices from Prof. Helder's lecture:
# - Compression: Delta uses Parquet under the hood (~80% compression)
# - Partitioning: by year of reference date → faster queries for time-filtered analyses
# - Lazy execution: all transformations chained, materialized only at saveAsTable()
#
# Note: DatCompetencia is StringType in Bronze, so we cast to date only
# for the partition column extraction (without modifying the source column).

# Create partition column from DatCompetencia (cast string to date temporarily)
df_samp_bronze_partitioned = df_samp_bronze.withColumn(
    "_partition_year",
    F.year(F.to_date(F.col("DatCompetencia"), "yyyy-MM-dd"))
)

print(f"💾 Writing to {TABLE_SAMP}...")
print(f"   📊 Partitioning by: _partition_year (extracted from DatCompetencia)")
print(f"   ⏳ Expected time: 1-3 minutes for ~1.5 GB\n")

(
    df_samp_bronze_partitioned.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_partition_year")
    .saveAsTable(TABLE_SAMP)
)

print(f"✅ SAMP Bronze table successfully created with partitioning!")
print(f"   📍 Location: {TABLE_SAMP}")
print(f"   🗂️  Partitions: by year (_partition_year)")
print(f"   🔍 Query it with: SELECT * FROM {TABLE_SAMP} LIMIT 10")

💾 Writing to energy_project.bronze.samp...
   📊 Partitioning by: _partition_year (extracted from DatCompetencia)
   ⏳ Expected time: 1-3 minutes for ~1.5 GB

✅ SAMP Bronze table successfully created with partitioning!
   📍 Location: energy_project.bronze.samp
   🗂️  Partitions: by year (_partition_year)
   🔍 Query it with: SELECT * FROM energy_project.bronze.samp LIMIT 10


In [0]:
# =============================================================================
# VALIDATE THE BRONZE TABLE
# =============================================================================
# Comprehensive validation after writing:
# - Table accessibility via Unity Catalog
# - Row count and schema
# - Partition structure (lesson from Prof. Helder)
# - Data sanity check (top distributors)

print("=" * 70)
print(f"🔍 VALIDATING: {TABLE_SAMP}")
print("=" * 70)

# Re-read from the catalog (proves it's queryable)
df_validation = spark.table(TABLE_SAMP)

# ----- 1. Basic info -----
print(f"\n📊 Table summary:")
print(f"   • Rows:    {df_validation.count():,}")
print(f"   • Columns: {len(df_validation.columns)}")

# ----- 2. Schema -----
print(f"\n📋 Final schema:")
df_validation.printSchema()

# ----- 3. Partitions check -----
print(f"\n🗂️  Partitions:")
df_validation.select("_partition_year").distinct().orderBy("_partition_year").show()

# ----- 4. Date range -----
print(f"📅 Reference date range (DatCompetencia):")
df_validation.agg(
    F.min("DatCompetencia").alias("min_date"),
    F.max("DatCompetencia").alias("max_date")
).show()

# ----- 5. Top distributors (sanity check) -----
print(f"🏢 Top 10 distributors by row count:")
(
    df_validation.groupBy("SigAgenteDistribuidora")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False)
)

print("=" * 70)
print(f"✅ VALIDATION COMPLETE — Bronze layer ready for Silver processing!")
print("=" * 70)

🔍 VALIDATING: energy_project.bronze.samp

📊 Table summary:
   • Rows:    6,808,591
   • Columns: 22

📋 Final schema:
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- NumCNPJAgenteDistribuidora: string (nullable = true)
 |-- SigAgenteDistribuidora: string (nullable = true)
 |-- NomAgenteDistribuidora: string (nullable = true)
 |-- NomTipoMercado: string (nullable = true)
 |-- DscModalidadeTarifaria: string (nullable = true)
 |-- DscSubGrupoTarifario: string (nullable = true)
 |-- DscClasseConsumoMercado: string (nullable = true)
 |-- DscSubClasseConsumidor: string (nullable = true)
 |-- DscDetalheConsumidor: string (nullable = true)
 |-- IdeAgenteAcessante: string (nullable = true)
 |-- NumCNPJAgenteAcessante: string (nullable = true)
 |-- NomAgenteAcessante: string (nullable = true)
 |-- DscPostoTarifario: string (nullable = true)
 |-- DscOpcaoEnergia: string (nullable = true)
 |-- DscDetalheMercado: string (nullable = true)
 |-- DatCompetencia: string (nullable = true)

In [0]:
# =============================================================================
# INADIMPLENCIA — READ CSV WITH EXPLICIT STRING SCHEMA
# =============================================================================

df_inadimp_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(f"{PATH_INADIMPLENCIA}/inadimplencia.csv")
)

print(f"✅ inadimplencia raw loaded")
print(f"   📊 Columns: {len(df_inadimp_raw.columns)}")
print(f"\n📋 Schema:")
df_inadimp_raw.printSchema()
print(f"\n🔍 Sample row:")
df_inadimp_raw.show(1, vertical=True, truncate=False)

✅ inadimplencia raw loaded
   📊 Columns: 7

📋 Schema:
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- SigAgente: string (nullable = true)
 |-- NumCNPJ: string (nullable = true)
 |-- SigIndicador: string (nullable = true)
 |-- AnoIndice: string (nullable = true)
 |-- NumPeriodoIndice: string (nullable = true)
 |-- VlrIndiceEnviado: string (nullable = true)


🔍 Sample row:
-RECORD 0---------------------------------
 DatGeracaoConjuntoDados | 05-05-2026     
 SigAgente               | EQUATORIAL PI  
 NumCNPJ                 | 06840748000189 
 SigIndicador            | ITotCrt        
 AnoIndice               | 2012           
 NumPeriodoIndice        | 1              
 VlrIndiceEnviado        | 2884,00        
only showing top 1 row


In [0]:
# =============================================================================
# INADIMPLENCIA — ADD METADATA + WRITE TO BRONZE
# =============================================================================

df_inadimp_bronze = (
    df_inadimp_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingestion_date", F.current_date())
)

print(f"💾 Writing to {TABLE_INADIMPLENCIA} ...")

(
    df_inadimp_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_INADIMPLENCIA)
)

print(f"✅ bronze.inadimplencia written successfully!")
print(f"   📍 Location: {TABLE_INADIMPLENCIA}")

💾 Writing to energy_project.bronze.inadimplencia ...
✅ bronze.inadimplencia written successfully!
   📍 Location: energy_project.bronze.inadimplencia


In [0]:
# =============================================================================
# DOMINIO INDICADORES — READ + WRITE TO BRONZE
# =============================================================================

df_dominio_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(f"{PATH_INADIMPLENCIA}/dominio-indicadores.csv")
)

print(f"📋 Schema:")
df_dominio_raw.printSchema()
print(f"\n🔍 All rows (small table):")
df_dominio_raw.show(truncate=False)

# Add metadata + write
df_dominio_bronze = (
    df_dominio_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingestion_date", F.current_date())
)

print(f"\n💾 Writing to {TABLE_DOMAIN_INDICATORS} ...")

(
    df_dominio_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_DOMAIN_INDICATORS)
)

print(f"✅ bronze.dominio_indicadores written successfully!")

📋 Schema:
root
 |-- DatGeracaoConjuntoDados: string (nullable = true)
 |-- SigIndicador: string (nullable = true)
 |-- DscIndicador: string (nullable = true)


🔍 All rows (small table):
+-----------------------+------------+--------------------------------------------------------------------------------+
|DatGeracaoConjuntoDados|SigIndicador|DscIndicador                                                                    |
+-----------------------+------------+--------------------------------------------------------------------------------+
|2023-02-05             |AREA        |Área do conj., expressa em km2, correspondente a área geogr. e não a área elétr.|
|2023-02-05             |AREAT       |Área do conjunto em km2                                                         |
|2023-02-05             |CM          |Encargo de uso do sistema de distribuição aplicado à unidade cons. (mensal)     |
|2023-02-05             |CMA         |Encargo de uso do sistema de distribuição aplicado à uni